# ブレーキング分析：圧力、リリース＆バランス

このノートブックでは、ブレーキングテクニックとコーナーでのGユーティライゼーションを詳細に分析します。

## このノートブックの内容

- **ピークブレーキ圧の一貫性**: 毎ラップ同じピーク圧を掛けているか？
- **進入速度の一貫性**: ブレーキングポイントでの速度 — 同じ速度で到達しているか？
- **ブレーキング距離の一貫性**: コーナーごとの制動距離のばらつき
- **ブレーキリリースポイントの一貫性**: ブレーキを離す位置 — トレイルブレーキングの鍵
- **ブレーキバランス分析**: コーナーごとのフロント/リアのブレーキバイアス（F/R別チャンネル必要）
- **Gユーティライゼーション分析**: コーナーコンプレックスでグリップをどれだけ連続的に使用しているか
- **サマリー統計テーブル**: 全ブレーキング指標を一覧表示

## 結果の解釈方法

- **狭い箱ひげ図**: 一貫したテクニック — 毎ラップ同じブレーキングをしている
- **広い箱ひげ図**: 一貫性がない — 改善の余地あり
- **Gユーティライゼーションが100%に近い**: スムーズな遷移、グリップの無駄が最小
- **Gユーティライゼーションが70%未満**: 遷移時の大きなグリップホール（優先コーチングエリア）

## 必要なチャンネル

- GPSデータチャンネル（`GPS Latitude`、`GPS Longitude`、`GPS Speed`）
- ブレーキ圧（`BrakePress`）とスロットル（`PPS`）
- Gユーティライゼーション分析用の横加速度（`LateralAcc`）
- バランス分析用のフロント/リア別ブレーキチャンネル（オプション）

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [1]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q pandas plotly libxrk libibt motorsports-data-notebook jinja2 ipywidgets

# Rustパーサーバックエンドを使用（ファイル読み込みが約3倍高速）
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# コアライブラリをインポート
import numpy as np
import pandas as pd
from IPython.display import display

# 可視化ライブラリ
import plotly.express as px
import plotly.graph_objects as go

# ヘルパー関数をインポート
from motorsports_data_notebook.channels import (
    get_best_lap_channels,
    get_top_laps,
)
from motorsports_data_notebook.corners import identify_corners
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_track_segments,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker
from motorsports_data_notebook.zones import (
    compute_g_utilization,
    compute_segment_stats,
    create_track_segments,
    detect_zones_averaged,
)

# セッションピッカーとチャンネル設定
# 自分の.xrk/.xrz/.ibtファイルをアップロードするか、サンプルデータを使用
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        # GPSチャンネル（コーナー検出に必要）
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "gps_speed": "GPS Speed",  # GPSからの速度（m/s）
        # ペダル入力（ゾーン検出に必要）
        "throttle": "PPS",  # スロットルポジションセンサー（0-100%）
        "brake": "BrakePress",  # ブレーキ圧（0-100%）
        # 車両ダイナミクス（Gユーティライゼーション分析に必要）
        "lateral_g": "LateralAcc",  # 横G
        "steering": "SteerAngle",  # ステアリング角度（度）
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 読み込んだセッションデータを取得
log = session.get_log()
laps = session.get_laps()
CHANNEL_NAMES = session.get_channel_names()

# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_time
0,1,150454,279602,2:09.148
1,2,279602,406240,2:06.638
2,3,406240,532797,2:06.557
3,4,532797,659283,2:06.486
4,5,659283,787773,2:08.490
5,6,787773,913776,2:06.003
6,7,913776,1041398,2:07.622
7,8,1041398,1168323,2:06.925
8,9,1168323,1294676,2:06.353
9,10,1294676,1420573,2:05.897


In [3]:
# ベストラップのチャンネルデータを抽出してコーナーを検出
best_lap, channels = get_best_lap_channels(
    log, laps, [CHANNEL_NAMES["gps_latitude"], CHANNEL_NAMES["gps_longitude"], "distance_m"]
)

best_lap_num = int(best_lap["num"])
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "distance_m"])
    .resample_to_channel(gps_lat_ch)
    .channels
)

lap_channels = {
    "GPS Latitude": aligned[gps_lat_ch].column(gps_lat_ch).to_numpy(),
    "GPS Longitude": aligned[gps_lon_ch].column(gps_lon_ch).to_numpy(),
    "distance_m": aligned["distance_m"].column("distance_m").to_numpy(),
}

corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.003,
    min_corner_length=15,
    min_gap=80,
)
print(f"{len(corners)}個のコーナーを検出")

10個のコーナーを検出


In [4]:
# 上位ラップを取得してゾーンを検出
top_laps = get_top_laps(laps, threshold_pct=1.03)
braking_zones, accel_zones = detect_zones_averaged(log, top_laps, CHANNEL_NAMES)

track_length = lap_channels["distance_m"][-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

# 各ラップのセグメント統計を計算
stats_df = compute_segment_stats(log, top_laps, segments, CHANNEL_NAMES)

print(f"{len(top_laps)}ラップを分析中（ベストタイムの103%以内）")
print(f"{len(braking_zones)}個のブレーキングゾーン、{len(accel_zones)}個の加速ゾーンを検出")
print(f"全ラップで{len(stats_df)}個のセグメント統計を計算")

13ラップを分析中（ベストタイムの103%以内）
7個のブレーキングゾーン、9個の加速ゾーンを検出
全ラップで390個のセグメント統計を計算


In [5]:
# ピークブレーキ圧の一貫性
# ドライバーが毎ラップ同じピークブレーキ圧を掛けているかを表示

braking_with_peak = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["peak_brake"])

if len(braking_with_peak) > 0:
    fig = px.box(
        braking_with_peak,
        x="segment_name",
        y="peak_brake",
        title="コーナー別ピークブレーキ圧の一貫性",
        labels={"peak_brake": "ピークブレーキ圧", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("ピークブレーキ圧データがありません")

In [6]:
# 進入速度の一貫性
# ブレーキングポイントでの速度 — 進入速度のばらつき＋一貫したブレーキングポイント = ブレーキングゾーン手前での速度変動

braking_with_entry = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["entry_speed"])

if len(braking_with_entry) > 0:
    fig = px.box(
        braking_with_entry,
        x="segment_name",
        y="entry_speed",
        title="コーナー別進入速度の一貫性",
        labels={"entry_speed": "進入速度 (km/h)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("進入速度データがありません")

In [7]:
# ブレーキング距離の一貫性
# コーナーごとの制動距離のばらつきを表示

braking_with_dist = stats_df[stats_df["segment_type"] == "braking"].dropna(
    subset=["braking_distance"]
)

if len(braking_with_dist) > 0:
    fig = px.box(
        braking_with_dist,
        x="segment_name",
        y="braking_distance",
        title="コーナー別ブレーキング距離の一貫性",
        labels={"braking_distance": "ブレーキング距離 (m)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("ブレーキング距離データがありません")

In [8]:
# ブレーキリリースポイントの一貫性
# ドライバーがブレーキを離す位置のばらつきを表示

braking_with_release = stats_df[stats_df["segment_type"] == "braking"].dropna(
    subset=["brake_release_point"]
)

if len(braking_with_release) > 0:
    braking_with_release = braking_with_release.copy()
    braking_with_release["release_deviation"] = braking_with_release.groupby("segment_name")[
        "brake_release_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        braking_with_release,
        x="segment_name",
        y="release_deviation",
        title="コーナー別ブレーキリリースポイントの一貫性（平均を中心に）",
        labels={"release_deviation": "平均からの偏差 (m)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("ブレーキリリースポイントデータがありません")

In [9]:
# ブレーキバランス分析（フロント/リア別のブレーキチャンネルがある場合のみ）
brake_rear_ch = CHANNEL_NAMES.get("brake_rear", "")
brake_front_ch = CHANNEL_NAMES["brake"]

if brake_rear_ch:
    balance_data = []
    for idx, lap in top_laps.iterrows():
        lap_num = int(lap["num"])
        try:
            aligned = (
                log.filter_by_lap(lap_num)
                .select_channels(["distance_m", brake_front_ch, brake_rear_ch])
                .resample_to_channel("distance_m")
                .channels
            )
        except Exception:
            continue

        dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
        front_arr = aligned[brake_front_ch].column(brake_front_ch).to_numpy()
        rear_arr = aligned[brake_rear_ch].column(brake_rear_ch).to_numpy()

        for seg in segments:
            if seg.segment_type != "braking":
                continue
            mask = (dist_arr >= seg.start_dist) & (dist_arr <= seg.end_dist)
            if not mask.any():
                continue
            front_peak = float(np.max(front_arr[mask]))
            rear_peak = float(np.max(rear_arr[mask]))
            total = front_peak + rear_peak
            if total > 0:
                balance_data.append(
                    {
                        "segment_name": seg.name,
                        "corner_id": seg.corner_id,
                        "lap_num": lap_num,
                        "front_peak": front_peak,
                        "rear_peak": rear_peak,
                        "front_bias_pct": front_peak / total * 100,
                    }
                )

    if balance_data:
        balance_df = pd.DataFrame(balance_data)

        # コーナー別フロント vs リアのピークブレーキ圧
        balance_grouped = (
            balance_df.groupby("segment_name")[["front_peak", "rear_peak"]].mean().reset_index()
        )
        fig = go.Figure()
        fig.add_trace(
            go.Bar(
                name="フロント",
                x=balance_grouped["segment_name"],
                y=balance_grouped["front_peak"],
            )
        )
        fig.add_trace(
            go.Bar(name="リア", x=balance_grouped["segment_name"], y=balance_grouped["rear_peak"])
        )
        fig.update_layout(
            barmode="group",
            title="コーナー別フロント vs リアのピークブレーキ圧",
            xaxis_title="コーナー",
            yaxis_title="ピークブレーキ圧",
            xaxis_tickangle=-45,
            width=900,
            height=500,
        )
        show_fig(fig)

        # フロントバイアス%の箱ひげ図
        fig = px.box(
            balance_df,
            x="segment_name",
            y="front_bias_pct",
            title="コーナー別フロントブレーキバイアス (%)",
            labels={"front_bias_pct": "フロントバイアス (%)", "segment_name": "コーナー"},
        )
        fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
        fig.add_hline(
            y=50,
            line_dash="dash",
            line_color="gray",
            opacity=0.5,
            annotation_text="50% = 均等バランス",
        )
        show_fig(fig)

        # 全体サマリー
        print(
            f"全体のフロントブレーキバイアス: {balance_df['front_bias_pct'].mean():.1f}% \u00b1 {balance_df['front_bias_pct'].std():.1f}%"
        )
    else:
        print("ブレーキバランスデータを計算できませんでした")
else:
    print("リアブレーキチャンネル未設定 \u2014 ブレーキバランス分析スキップ")

リアブレーキチャンネル未設定 — ブレーキバランス分析スキップ


In [10]:
# ブレーキバランスコースマップ（リアブレーキチャンネルがある場合のみ）
if brake_rear_ch:
    try:
        aligned = (
            log.filter_by_lap(best_lap_num)
            .select_channels(["distance_m", brake_front_ch, brake_rear_ch])
            .resample_to_channel("distance_m")
            .channels
        )
        dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
        front_arr = aligned[brake_front_ch].column(brake_front_ch).to_numpy()
        rear_arr = aligned[brake_rear_ch].column(brake_rear_ch).to_numpy()

        total_brake = front_arr + rear_arr
        braking_mask = total_brake > np.max(total_brake) * 0.05
        balance_pct = np.where(
            braking_mask, front_arr / np.maximum(total_brake, 1e-6) * 100, np.nan
        )

        gps_dist = lap_channels["distance_m"]
        braking_indices = np.where(braking_mask)[0]
        if len(braking_indices) > 0:
            braking_dists = dist_arr[braking_indices]
            braking_balance = balance_pct[braking_indices]
            lat_interp = np.interp(braking_dists, gps_dist, lap_channels["GPS Latitude"])
            lon_interp = np.interp(braking_dists, gps_dist, lap_channels["GPS Longitude"])

            fig = go.Figure()
            fig.add_trace(
                go.Scattermapbox(
                    lat=lap_channels["GPS Latitude"],
                    lon=lap_channels["GPS Longitude"],
                    mode="markers",
                    marker=dict(size=3, color="lightgray"),
                    name="コース",
                    showlegend=False,
                )
            )
            fig.add_trace(
                go.Scattermapbox(
                    lat=lat_interp,
                    lon=lon_interp,
                    mode="markers",
                    marker=dict(
                        size=8,
                        color=braking_balance,
                        colorscale="RdYlBu_r",
                        showscale=True,
                        colorbar=dict(title="フロントバイアス %"),
                        cmin=40,
                        cmax=80,
                    ),
                    name="ブレーキバランス",
                )
            )
            fig.update_layout(
                mapbox=dict(
                    style="open-street-map",
                    center=dict(
                        lat=np.mean(lap_channels["GPS Latitude"]),
                        lon=np.mean(lap_channels["GPS Longitude"]),
                    ),
                    zoom=14,
                ),
                title="ブレーキバランスコースマップ（ベストラップ）",
                showlegend=False,
                width=800,
                height=600,
            )
            show_fig(fig)
        else:
            print("ベストラップで有意なブレーキングが検出されませんでした")
    except Exception as e:
        print(f"ブレーキバランスコースマップを計算できませんでした: {e}")
else:
    print("リアブレーキチャンネル未設定 \u2014 ブレーキバランスコースマップスキップ")

リアブレーキチャンネル未設定 — ブレーキバランスコースマップスキップ


## Gユーティライゼーション分析

Gユーティライゼーションは、ブレーキング \u2192 ターンイン \u2192 ミッドコーナー \u2192 脱出 \u2192 加速の一連の動作を通じて、ドライバーがタイヤのグリップをどれだけ連続的に使用しているかを測定します。トータルG（= \u221a(横G\u00b2 + 縦G\u00b2)）の低下はグリップの無駄を示します。

- **高いg_utilization_pct**（100%に近い）: スムーズな遷移、グリップの無駄が最小
- **低いg_utilization_pct**: 遷移時の「グリップホール」 \u2014 ドライバーがフェーズ間で停滞
- **total_g_min_phase**: グリップホールが発生するフェーズ \u2014 ドライバーが*どこで*躊躇しているかを示す

横Gデータが必要です。縦Gが利用できない場合は速度から導出されます。

In [11]:
# コーナーごと・ラップごとのGユーティライゼーションを計算
# 横Gが必要；縦Gが利用できない場合は速度から導出

lateral_g_ch = CHANNEL_NAMES["lateral_g"]
inline_g_ch = CHANNEL_NAMES.get("inline_g", "")
gps_speed_ch = CHANNEL_NAMES["gps_speed"]

g_util_channels = ["distance_m", gps_speed_ch, lateral_g_ch]
if inline_g_ch:
    g_util_channels.append(inline_g_ch)

distances_list: list[np.ndarray] = []
speeds_list: list[np.ndarray] = []
lateral_gs_list: list[np.ndarray] = []
inline_gs_list: list[np.ndarray] | None = [] if inline_g_ch else None
lap_nums_list: list[int] = []

for idx, lap in top_laps.iterrows():
    lap_num = int(lap["num"])
    try:
        aligned = (
            log.filter_by_lap(lap_num)
            .select_channels(g_util_channels)
            .resample_to_channel("distance_m")
            .channels
        )
    except Exception:
        continue

    dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
    speed_arr = aligned[gps_speed_ch].column(gps_speed_ch).to_numpy() * 3.6  # m/s -> km/h
    lat_g_arr = aligned[lateral_g_ch].column(lateral_g_ch).to_numpy()

    distances_list.append(dist_arr)
    speeds_list.append(speed_arr)
    lateral_gs_list.append(lat_g_arr)
    lap_nums_list.append(lap_num)

    if inline_g_ch and inline_gs_list is not None:
        inline_gs_list.append(aligned[inline_g_ch].column(inline_g_ch).to_numpy())

if len(distances_list) > 0:
    g_util_df = compute_g_utilization(
        distances=distances_list,
        speeds=speeds_list,
        lateral_gs=lateral_gs_list,
        inline_gs=inline_gs_list,
        lap_nums=lap_nums_list,
        segments=segments,
        corners=corners,
    )
    print(f"{len(g_util_df)}個のコーナー/ラップ組み合わせでGユーティライゼーションを計算")
else:
    g_util_df = pd.DataFrame()
    print("Gユーティライゼーション分析に使用できる横Gデータがありません")

130個のコーナー/ラップ組み合わせでGユーティライゼーションを計算


In [12]:
# コーナー別Gユーティライゼーション — 箱ひげ図
# 各コーナーコンプレックスでドライバーがグリップをどれだけ一貫して維持しているかを表示

if len(g_util_df) > 0:
    corner_order = [c.name for c in corners if c.name in g_util_df["corner_name"].values]
    fig = px.box(
        g_util_df,
        x="corner_name",
        y="g_utilization_pct",
        title="コーナー別Gユーティライゼーション（高いほどスムーズな遷移）",
        labels={
            "g_utilization_pct": "Gユーティライゼーション (%)",
            "corner_name": "コーナー",
        },
        category_orders={"corner_name": corner_order},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    fig.add_hline(
        y=70,
        line_dash="dash",
        line_color="orange",
        opacity=0.5,
        annotation_text="70% = 大きなグリップホール",
    )
    show_fig(fig)
else:
    print("Gユーティライゼーションデータがありません")

In [13]:
# フェーズ別G内訳 — グループ棒グラフ
# コーナーごとのフェーズ別平均トータルGを表示、グリップの弱い部分を強調

if len(g_util_df) > 0:
    phase_cols = ["braking_g_mean", "entry_g_mean", "mid_g_mean", "exit_g_mean"]
    phase_labels = ["ブレーキング", "進入", "ミッドコーナー", "脱出"]

    # コーナーごとにラップ間で平均化
    phase_avg = g_util_df.groupby("corner_name")[phase_cols].mean().reset_index()

    corner_order = [c.name for c in corners if c.name in phase_avg["corner_name"].values]
    phase_avg = phase_avg.set_index("corner_name").loc[corner_order].reset_index()

    fig = go.Figure()
    colors = ["#EF553B", "#FFA15A", "#00CC96", "#636EFA"]
    for col, label, color in zip(phase_cols, phase_labels, colors):
        fig.add_trace(
            go.Bar(
                name=label,
                x=phase_avg["corner_name"],
                y=phase_avg[col],
                marker_color=color,
            )
        )

    fig.update_layout(
        barmode="group",
        title="コーナー別フェーズごとの平均トータルG（最低フェーズ = グリップホール）",
        xaxis_title="コーナー",
        yaxis_title="平均トータルG",
        xaxis_tickangle=-45,
        width=900,
        height=500,
    )
    show_fig(fig)
else:
    print("フェーズ別G内訳に使用できるGユーティライゼーションデータがありません")

In [14]:
# トータルGトレース — ベストラップのコーナーシェーディングとGホールマーカー付き
# ベストラップのトータルG vs 距離をコーナー領域ハイライト付きで表示

if len(g_util_df) > 0:
    gps_speed_ch = CHANNEL_NAMES["gps_speed"]

    bl_channels = ["distance_m", gps_speed_ch, lateral_g_ch]
    if inline_g_ch:
        bl_channels.append(inline_g_ch)

    bl_aligned = (
        log.filter_by_lap(best_lap_num)
        .select_channels(bl_channels)
        .resample_to_channel("distance_m")
        .channels
    )

    bl_dist = bl_aligned["distance_m"].column("distance_m").to_numpy()
    bl_speed = bl_aligned[gps_speed_ch].column(gps_speed_ch).to_numpy() * 3.6
    bl_lat_g = bl_aligned[lateral_g_ch].column(lateral_g_ch).to_numpy()

    if inline_g_ch:
        bl_inl_g = bl_aligned[inline_g_ch].column(inline_g_ch).to_numpy()
    else:
        # 速度から導出
        spd_ms = bl_speed / 3.6
        dd = np.diff(bl_dist)
        avg_spd = (spd_ms[:-1] + spd_ms[1:]) / 2
        safe_avg = np.where(avg_spd > 0.1, avg_spd, 0.1)
        dt = dd / safe_avg
        dv = np.diff(spd_ms)
        safe_dt = np.where(dt > 1e-6, dt, 1e-6)
        accel = dv / safe_dt / 9.81
        accel = np.concatenate([[accel[0]], accel])
        kernel = np.ones(5) / 5
        bl_inl_g = np.convolve(accel, kernel, mode="same")

    bl_total_g = np.sqrt(bl_lat_g**2 + bl_inl_g**2)

    fig = go.Figure()

    # コーナーシェーディングを追加
    corner_colors = ["rgba(100,100,255,0.1)", "rgba(255,100,100,0.1)"]
    for i, corner in enumerate(corners):
        fig.add_vrect(
            x0=corner.start_dist,
            x1=corner.end_dist,
            fillcolor=corner_colors[i % 2],
            layer="below",
            line_width=0,
            annotation_text=corner.name,
            annotation_position="top left",
            annotation_font_size=9,
        )

    # トータルGトレース
    fig.add_trace(
        go.Scatter(
            x=bl_dist,
            y=bl_total_g,
            mode="lines",
            name="トータルG",
            line=dict(color="#636EFA", width=1.5),
        )
    )

    # ベストラップのg_util_dfからGホールマーカーを追加
    best_lap_g = g_util_df[g_util_df["lap_num"] == best_lap_num]
    if "total_g_min_dist" in best_lap_g.columns and len(best_lap_g) > 0:
        fig.add_trace(
            go.Scatter(
                x=best_lap_g["total_g_min_dist"],
                y=best_lap_g["total_g_min"],
                mode="markers+text",
                marker=dict(size=10, color="red", symbol="diamond"),
                text=[
                    f"{row['corner_name']}: {row['total_g_min']:.2f}G"
                    for _, row in best_lap_g.iterrows()
                ],
                textposition="top center",
                textfont=dict(size=9, color="red"),
                name="Gホール",
            )
        )

    fig.update_layout(
        title=f"トータルGトレース — ベストラップ（ラップ {best_lap_num}）",
        xaxis_title="距離 (m)",
        yaxis_title="トータルG",
        width=1100,
        height=400,
        showlegend=False,
    )
    show_fig(fig)
else:
    print("トータルGトレースに使用できるGユーティライゼーションデータがありません")

In [15]:
# Gホール検出 — 遷移時にトータルGが大きく低下するコーナーを特定
# Gホールはブレーキングとターンインのスムーズなオーバーラップができていない
# （トレイルブレーキングのギャップ）またはターンと加速の間の躊躇を示す

if len(g_util_df) > 0 and "total_g_min_dist" in g_util_df.columns:
    # コーナーごとにラップ間のGホール指標を平均化
    g_hole_summary = (
        g_util_df.groupby("corner_name")
        .agg(
            g_min_mean=("total_g_min", "mean"),
            g_min_std=("total_g_min", "std"),
            g_max_mean=("total_g_max", "mean"),
            g_util_mean=("g_utilization_pct", "mean"),
            g_min_dist_mean=("total_g_min_dist", "mean"),
        )
        .reset_index()
    )

    # Gホールの深さ：最小値がピークからどれだけ落ちるか
    g_hole_summary["g_hole_depth"] = g_hole_summary["g_max_mean"] - g_hole_summary["g_min_mean"]

    # 各コーナーの最頻フェーズを決定
    from collections import Counter

    phase_map = {}
    for name, group in g_util_df.groupby("corner_name"):
        phases = group["total_g_min_phase"].dropna().values
        if len(phases) > 0:
            phase_map[name] = Counter(phases).most_common(1)[0][0]
    g_hole_summary["g_hole_phase"] = g_hole_summary["corner_name"].map(phase_map)

    # Gホールの深さで並べ替え（最悪が先）
    g_hole_summary = g_hole_summary.sort_values("g_hole_depth", ascending=False)

    # 重大なGホールをフラグ
    g_hole_summary["severity"] = g_hole_summary["g_util_mean"].apply(
        lambda x: "高" if x < 30 else ("中" if x < 50 else ("低" if x < 70 else "OK"))
    )

    # フェーズの解釈
    phase_advice = {
        "braking": "ブレーキング中のGホール — 遅い/急なブレーキ操作",
        "entry": "トレイルブレーキングのギャップ — ターンインで横Gが立ち上がる前にブレーキをリリース",
        "mid": "ミッドコーナーの躊躇 — エイペックスでリフトまたは停滞",
        "exit": "脱出の躊躇 — ターンから加速への切り替えにギャップ",
    }
    g_hole_summary["interpretation"] = g_hole_summary["g_hole_phase"].map(phase_advice)

    # 表示
    display_cols = [
        "corner_name",
        "g_min_mean",
        "g_max_mean",
        "g_hole_depth",
        "g_util_mean",
        "g_hole_phase",
        "severity",
        "interpretation",
    ]
    display_df = g_hole_summary[display_cols].rename(
        columns={
            "corner_name": "コーナー",
            "g_min_mean": "最小G (平均)",
            "g_max_mean": "最大G (平均)",
            "g_hole_depth": "Gホール深さ",
            "g_util_mean": "Gユーティライゼーション %",
            "g_hole_phase": "フェーズ",
            "severity": "重要度",
            "interpretation": "解釈",
        }
    )

    print("Gホール検出サマリー（深さ順、最悪が先）:")
    display(
        display_df.style.format(
            {
                "最小G (平均)": "{:.2f}",
                "最大G (平均)": "{:.2f}",
                "Gホール深さ": "{:.2f}",
                "Gユーティライゼーション %": "{:.1f}",
            }
        )
    )

    # 高重要度のコーナーを警告
    high = g_hole_summary[g_hole_summary["severity"] == "高"]
    if len(high) > 0:
        print(f"\n⚠ 高重要度のGホールがある{len(high)}個のコーナー:")
        for _, row in high.iterrows():
            print(
                f"  {row['corner_name']}: Gが{row['g_min_mean']:.2f}Gまで低下 "
                f"（ピーク{row['g_max_mean']:.2f}Gから）{row['g_hole_phase']}フェーズ"
            )
else:
    print("Gホール検出に使用できるGユーティライゼーションデータがありません")

Gホール検出サマリー（深さ順、最悪が先）:


,コーナー,最小G (平均),最大G (平均),Gホール深さ,Gユーティライゼーション %,フェーズ,重要度,解釈
3,Turn 3,0.14,1.46,1.31,11.3,braking,高,ブレーキング中のGホール — 遅い/急なブレーキ操作
1,Turn 10,0.14,1.42,1.28,11.9,braking,高,ブレーキング中のGホール — 遅い/急なブレーキ操作
9,Turn 9,0.21,1.47,1.26,15.9,braking,高,ブレーキング中のGホール — 遅い/急なブレーキ操作
8,Turn 8,0.24,1.41,1.17,18.3,braking,高,ブレーキング中のGホール — 遅い/急なブレーキ操作
7,Turn 7,0.33,1.37,1.04,25.3,braking,高,ブレーキング中のGホール — 遅い/急なブレーキ操作
4,Turn 4,0.61,1.52,0.91,57.3,entry,低,トレイルブレーキングのギャップ — ターンインで横Gが立ち上がる前にブレーキをリリース
2,Turn 2,0.76,1.52,0.76,88.6,entry,OK,トレイルブレーキングのギャップ — ターンインで横Gが立ち上がる前にブレーキをリリース
0,Turn 1,0.61,1.27,0.66,54.8,braking,低,ブレーキング中のGホール — 遅い/急なブレーキ操作
6,Turn 6,0.74,1.37,0.63,64.2,braking,低,ブレーキング中のGホール — 遅い/急なブレーキ操作
5,Turn 5,0.36,0.89,0.53,60.3,braking,低,ブレーキング中のGホール — 遅い/急なブレーキ操作



⚠ 高重要度のGホールがある5個のコーナー:
  Turn 3: Gが0.14Gまで低下 （ピーク1.46Gから）brakingフェーズ
  Turn 10: Gが0.14Gまで低下 （ピーク1.42Gから）brakingフェーズ
  Turn 9: Gが0.21Gまで低下 （ピーク1.47Gから）brakingフェーズ
  Turn 8: Gが0.24Gまで低下 （ピーク1.41Gから）brakingフェーズ
  Turn 7: Gが0.33Gまで低下 （ピーク1.37Gから）brakingフェーズ


In [16]:
# ブレーキング指標のサマリー統計テーブル
def compute_braking_summary(stats_df):
    """全ラップにわたるブレーキング指標のサマリー統計を計算"""
    summary = []
    metrics = [
        ("peak_brake", "ピークブレーキ圧"),
        ("entry_speed", "進入速度 (km/h)"),
        ("braking_distance", "ブレーキング距離 (m)"),
        ("brake_release_point", "ブレーキリリースポイント (m)"),
        ("braking_point", "ブレーキングポイント (m)"),
    ]

    for seg_name in stats_df[stats_df["segment_type"] == "braking"]["segment_name"].unique():
        for col, label in metrics:
            seg_data = stats_df[(stats_df["segment_name"] == seg_name) & stats_df[col].notna()]
            if len(seg_data) > 0:
                summary.append(
                    {
                        "セグメント": seg_name,
                        "指標": label,
                        "平均": seg_data[col].mean(),
                        "標準偏差": seg_data[col].std(),
                        "最小": seg_data[col].min(),
                        "最大": seg_data[col].max(),
                        "範囲": seg_data[col].max() - seg_data[col].min(),
                        "N": len(seg_data),
                    }
                )

    return pd.DataFrame(summary)


braking_summary_df = compute_braking_summary(stats_df)
if len(braking_summary_df) > 0:
    braking_summary_df.style.format(
        {
            "平均": "{:.1f}",
            "標準偏差": "{:.1f}",
            "最小": "{:.1f}",
            "最大": "{:.1f}",
            "範囲": "{:.1f}",
        }
    )
else:
    print("サマリー用のブレーキングデータがありません")